# 🧠 Local CPU-Only PDF-to-Instruction Dataset Pipeline

Welcome to the **End-to-End CPU PDF-to-Instruction Pipeline**.

This production-quality notebook transforms collections of raw, unstructured PDF books into meticulously cleaned, semantically segmented instruction datasets suitable for fine-tuning Small Language Models (SLMs).

## 1. Installation
This section ensures all required libraries are installed. We use actively maintained open-source libraries optimized for text processing and CPU execution.
- `PyMuPDF` (fitz) & `pdfplumber` & `docling`: Document extraction without OCR.
- `spacy` & `nltk`: Advanced NLP, sentence boundaries.
- `rapidfuzz`, `xxhash`: Deduplication and hashing.
- `transformers`, `datasets`: CPU Synthetic data & schemas.
- `orjson`, `pyarrow`: Fast serialization.


In [1]:
%pip install -q --break-system-packages PyMuPDF pdfplumber docling marker-pdf spacy nltk regex orjson rapidfuzz pandas numpy networkx tqdm rich sentencepiece transformers datasets pyarrow scikit-learn xxhash matplotlib beautifulsoup4 lxml joblib
!python3 -m spacy download en_core_web_sm -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

Note: you may need to restart the kernel to use updated packages.
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommen

True

## 2. Imports
Here we import all required libraries. The architecture follows object-oriented design, separating concerns into modular classes.

In [2]:
import os
import sys
import glob
import time
import json
import logging
import shutil
import unicodedata
import re
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Set
from dataclasses import dataclass, field, asdict
from datetime import datetime
import concurrent.futures

import fitz  # PyMuPDF
import pdfplumber
import spacy
import orjson
import xxhash
from rapidfuzz import fuzz
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import Dataset

import warnings
warnings.filterwarnings('ignore')

console = Console()

## 3. Configuration
The `PipelineConfig` dataclass acts as the single source of truth for the entire pipeline. It controls directory paths, chunking heuristics (size, overlap), deduplication thresholds, and system execution parameters (random seeds, parallel workers).

In [3]:
@dataclass
class PipelineConfig:
    # Base Directories
    base_dir: Path = Path('./pipeline_workspace')
    input_pdfs_dir: Path = Path('./pipeline_workspace/input_pdfs')
    
    # Dynamic Subdirectories
    output_dir: Path = field(init=False)
    cache_dir: Path = field(init=False)
    logs_dir: Path = field(init=False)
    metadata_dir: Path = field(init=False)
    clean_text_dir: Path = field(init=False)
    chunks_dir: Path = field(init=False)
    datasets_dir: Path = field(init=False)
    stats_dir: Path = field(init=False)
    reports_dir: Path = field(init=False)
    exports_dir: Path = field(init=False)
    config_dir: Path = field(init=False)
    
    # Chunking Hyperparameters
    target_chunk_size: int = 300      # Target number of words per chunk
    min_words_per_chunk: int = 40     # Drop chunks smaller than this
    max_words_per_chunk: int = 600    # Hard limit on chunk size
    chunk_overlap: int = 2            # Sentences to overlap between sequential chunks
    min_chars: int = 150              # Minimum character limit
    
    # Formatting & Cleaning
    language: str = 'en'
    sentence_separator: str = ' '
    duplicate_threshold: float = 90.0 # Rapidfuzz similarity threshold for near-duplicates (0-100)
    header_margin_pct: float = 0.08   # Ignore top 8% of page (headers)
    footer_margin_pct: float = 0.08   # Ignore bottom 8% of page (footers)
    
    # Execution Configuration
    log_level: int = logging.INFO
    random_seed: int = 42
    max_workers: int = max(1, (os.cpu_count() or 4) - 1)
    export_formats: List[str] = field(default_factory=lambda: ['jsonl', 'csv', 'parquet', 'json', 'arrow'])
    
    def __post_init__(self):
        self.output_dir = self.base_dir / 'output'
        self.cache_dir = self.base_dir / 'cache'
        self.logs_dir = self.base_dir / 'logs'
        self.metadata_dir = self.base_dir / 'metadata'
        self.clean_text_dir = self.base_dir / 'clean_text'
        self.chunks_dir = self.base_dir / 'chunks'
        self.datasets_dir = self.base_dir / 'datasets'
        self.stats_dir = self.base_dir / 'statistics'
        self.reports_dir = self.base_dir / 'reports'
        self.exports_dir = self.base_dir / 'exports'
        self.config_dir = self.base_dir / 'config'

config = PipelineConfig()

## 4. Utility Functions
This module provides fast hashing, robust logging, and environment directory initialization.

In [4]:
def setup_environment(cfg: PipelineConfig):
    """Creates all necessary directories safely."""
    directories = [
        cfg.input_pdfs_dir, cfg.output_dir, cfg.cache_dir, cfg.logs_dir,
        cfg.metadata_dir, cfg.clean_text_dir, cfg.chunks_dir,
        cfg.datasets_dir, cfg.stats_dir, cfg.reports_dir, 
        cfg.exports_dir, cfg.config_dir
    ]
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
        
    with open(cfg.config_dir / 'active_config.json', 'w') as f:
        json.dump(asdict(cfg), f, indent=4, default=str)
        
    console.print('[bold green]✓ Pipeline directories established.[/bold green]')

def setup_logger(name: str, cfg: PipelineConfig) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(cfg.log_level)
        formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
        cfg.logs_dir.mkdir(parents=True, exist_ok=True)
        file_handler = logging.FileHandler(cfg.logs_dir / 'pipeline_execution.log')
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
    return logger

logger = setup_logger('PDFPipeline', config)

def compute_file_hash(filepath: Path) -> str:
    """Generates xxhash for ultra-fast exact file deduplication."""
    h = xxhash.xxh64()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

## 5. PDF Discovery
Recursively scans folders to find PDFs. Uses `xxhash` to detect duplicate files, checks for encryption, and generates a discovery report.

In [5]:
class PDFScanner:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.registry = []
        
    def generate_mock_pdf(self):
        demo_path = self.cfg.input_pdfs_dir / 'demo_book.pdf'
        doc = fitz.open()
        p1 = doc.new_page()
        p1.insert_text((50, 50), "The AI Engineer's Handbook", fontsize=24)
        p1.insert_text((50, 120), "Chapter 1: The Foundations of Language Models", fontsize=16)
        p1.insert_text((50, 160), "Small Language Models (SLMs) represent a paradigm shift in artificial intelligence. " * 10, fontsize=11)
        doc.save(demo_path)
        doc.close()
        console.print('[yellow]ℹ️ Input directory was empty. Generated demo_book.pdf.[/yellow]')

    def scan(self) -> List[Dict]:
        pdfs = list(self.cfg.input_pdfs_dir.rglob("*.pdf"))
        
        # Add the user's target file
        target_file = Path('/Users/cemkaya/Documents/Ebooks_important/Practical Guide to LTE-A VoLTE and IoT Paving the way towards 5G.pdf')
        if target_file.exists():
            pdfs.append(target_file)
            
        if not pdfs:
            self.generate_mock_pdf()
            pdfs = list(self.cfg.input_pdfs_dir.rglob("*.pdf"))
            
        seen_hashes = set()
        
        for path in tqdm(pdfs, desc='Scanning PDFs'):
            file_hash = compute_file_hash(path)
            is_dup = file_hash in seen_hashes
            seen_hashes.add(file_hash)
            
            status = 'valid'
            try:
                with fitz.open(path) as doc:
                    if doc.is_encrypted:
                        status = 'encrypted'
            except Exception as e:
                status = 'corrupted'
                logger.error(f'Corrupted PDF: {path} | Error: {e}')
                
            self.registry.append({
                'path': str(path),
                'filename': path.name,
                'size_bytes': path.stat().st_size,
                'hash': file_hash,
                'status': status,
                'is_duplicate': is_dup
            })
            
        with open(self.cfg.metadata_dir / 'pdf_registry.json', 'wb') as f:
            f.write(orjson.dumps(self.registry, option=orjson.OPT_INDENT_2))
            
        valid = [p for p in self.registry if p['status'] == 'valid' and not p['is_duplicate']]
        console.print(f'[bold green]✓ Discovery: Found {len(valid)} valid unique PDFs out of {len(pdfs)} total.[/bold green]')
        return valid

## 6. Metadata Extraction
Extracts extensive metadata from the PDF (Title, Author, Creation date, Modification date, Producer, Page count, PDF version, File size, Document hash).

In [6]:
class MetadataExtractor:
    def extract(self, pdf_info: Dict) -> Dict:
        path = pdf_info['path']
        doc = fitz.open(path)
        meta = doc.metadata
        
        creation_date = meta.get('creationDate', '')
        mod_date = meta.get('modDate', '')
        
        extracted = {
            'book_title': meta.get('title', Path(path).stem) or Path(path).stem,
            'author': meta.get('author', 'Unknown') or 'Unknown',
            'creation_date': creation_date,
            'modification_date': mod_date,
            'producer': meta.get('producer', 'Unknown') or 'Unknown',
            'page_count': doc.page_count,
            'pdf_version': doc.metadata.get('format', 'PDF'),
            'file_size_bytes': pdf_info['size_bytes'],
            'document_hash': pdf_info['hash'],
            'file_name': pdf_info['filename']
        }
        doc.close()
        return extracted

## 7. Structure Extraction
Reads PyMuPDF `blocks`. By sorting block coordinates, reading order is strictly preserved. Infers structure (headings, tables, paragraphs).

In [7]:
class StructureExtractor:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        
    def extract(self, path: str) -> List[Dict]:
        doc = fitz.open(path)
        blocks_data = []
        
        for page_num in range(doc.page_count):
            page = doc.load_page(page_num)
            page_dict = page.get_text('dict')
            width = page_dict['width']
            height = page_dict['height']
            
            blocks = page.get_text('blocks')
            blocks.sort(key=lambda b: (b[1], b[0])) # Sort top-to-bottom, left-to-right
            
            for b in blocks:
                if b[6] != 0: continue # Skip images
                
                # Header / Footer rejection
                if b[1] < height * self.cfg.header_margin_pct: continue
                if b[3] > height * (1 - self.cfg.footer_margin_pct): continue
                
                raw_text = b[4]
                if len(raw_text.strip()) < 2: continue
                
                # Check TOC / Junk patterns (high density of numbers)
                words = raw_text.split()
                if words:
                    num_count = sum(1 for w in words if any(c.isdigit() for c in w))
                    if num_count / len(words) > 0.20:
                        continue
                
                block_type = 'paragraph'
                # Simple heuristics for headings and lists
                if raw_text.strip().startswith(('-', '•', '* ', '1.', 'a)')):
                    block_type = 'list'
                elif len(raw_text) < 150 and raw_text.strip().istitle() and not raw_text.strip().endswith(('.', ':', '?', '!')):
                    block_type = 'heading'
                    
                blocks_data.append({
                    'page': page_num + 1,
                    'type': block_type,
                    'raw_text': raw_text
                })
        doc.close()
        return blocks_data

## 8. Text Cleaning
Normalizes unicode, purges invisible control characters, and mends broken line-endings without losing paragraphs. Preserves math expressions and code blocks.

In [8]:
class TextCleaner:
    def clean(self, text: str) -> str:
        if not text:
            return ''
            
        # Strip invisible control characters (except basic whitespace)
        text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C' or ch in ['\n', '\t', '\r'])
        
        # Unicode Normalization
        text = unicodedata.normalize('NFKC', text)
        
        # Detect if it's a code block (simple heuristic: contains typical programming syntax spacing)
        is_code = '    ' in text or '\t' in text or 'def ' in text or 'function(' in text
        if is_code:
            return text.strip() # Preserve code block formatting exactly
            
        # Fix broken line endings (lines that break mid-sentence)
        lines = text.split('\n')
        fixed_text = ""
        for i, line in enumerate(lines):
            line = line.strip()
            if not line:
                fixed_text += "\n\n"
            else:
                fixed_text += line + " "
                
        # Final pass condense
        fixed_text = ' '.join(fixed_text.split())
        return fixed_text.strip()

## 9. Document Segmentation
Detects Chapters, Sections, Subsections. Every block and subsequently every chunk will know its parent section.

In [9]:
class DocumentSegmenter:
    def segment(self, blocks: List[Dict], cleaner: TextCleaner, root_title: str) -> List[Dict]:
        segmented_blocks = []
        current_chapter = "Root"
        current_section = "Root"
        current_subsection = "Root"
        
        for block in blocks:
            clean_t = cleaner.clean(block['raw_text'])
            if not clean_t: continue
            
            if block['type'] == 'heading':
                lower_t = clean_t.lower()
                if 'chapter' in lower_t:
                    current_chapter = clean_t
                    current_section = "Root"
                    current_subsection = "Root"
                elif len(clean_t.split()) < 7:
                    if current_section == "Root":
                        current_section = clean_t
                    else:
                        current_subsection = clean_t
                        
            segmented_blocks.append({
                'page': block['page'],
                'type': block['type'],
                'text': clean_t,
                'hierarchy': {
                    'chapter': current_chapter,
                    'section': current_section,
                    'subsection': current_subsection,
                    'path': f"{current_chapter} > {current_section} > {current_subsection}"
                }
            })
            
        return segmented_blocks

## 10. Semantic Chunking
Uses `spaCy` to identify precise sentence boundaries. Sentences are grouped until they hit the `target_chunk_size`, with configurable overlap.

In [10]:
class SemanticChunker:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        try:
            self.nlp = spacy.load('en_core_web_sm', disable=['ner', 'tagger', 'lemmatizer', 'textcat'])
            self.nlp.add_pipe('sentencizer')
        except OSError:
            logger.error("spaCy model missing.")
            raise

    def chunk(self, segmented_blocks: List[Dict]) -> List[Dict]:
        chunks = []
        current_sents = []
        current_word_count = 0
        current_pages = set()
        current_hierarchy = None
        
        for block in segmented_blocks:
            if block['type'] == 'heading':
                current_hierarchy = block['hierarchy']
                continue
                
            if not current_hierarchy:
                current_hierarchy = block['hierarchy']
                
            doc_spacy = self.nlp(block['text'])
            for sent in doc_spacy.sents:
                s_text = sent.text.strip()
                if not s_text: continue
                
                words = s_text.split()
                s_wc = len(words)
                
                if current_word_count + s_wc > self.cfg.target_chunk_size and current_word_count >= self.cfg.min_words_per_chunk:
                    chunks.append({
                        'text': ' '.join(current_sents),
                        'pages': list(current_pages),
                        'hierarchy': current_hierarchy,
                        'sentence_count': len(current_sents)
                    })
                    overlap = current_sents[-self.cfg.chunk_overlap:] if self.cfg.chunk_overlap > 0 else []
                    current_sents = overlap
                    current_word_count = sum(len(x.split()) for x in overlap)
                    current_pages = set()
                    
                current_sents.append(s_text)
                current_word_count += s_wc
                current_pages.add(block['page'])
                
        if current_sents and current_word_count >= self.cfg.min_words_per_chunk:
            chunks.append({
                'text': ' '.join(current_sents),
                'pages': list(current_pages),
                'hierarchy': current_hierarchy,
                'sentence_count': len(current_sents)
            })
            
        return chunks

## 11. Metadata Generation
Generates metadata for every chunk (Word count, Character count, Token estimates, Reading time). Generation of stable chunk IDs.

In [11]:
class MetadataGenerator:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        
    def generate(self, raw_chunks: List[Dict], doc_metadata: Dict) -> List[Dict]:
        enriched_chunks = []
        for c in raw_chunks:
            text = c['text']
            words = text.split()
            word_count = len(words)
            char_count = len(text)
            chunk_hash = xxhash.xxh64(text.encode('utf-8')).hexdigest()
            
            meta = {
                'book': doc_metadata['book_title'],
                'file_name': doc_metadata['file_name'],
                'page_range': c['pages'],
                'chapter': c['hierarchy']['chapter'],
                'section': c['hierarchy']['section'],
                'subsection': c['hierarchy']['subsection'],
                'hierarchy_path': c['hierarchy']['path'],
                'chunk_id': f"chk_{chunk_hash}",
                'chunk_hash': chunk_hash,
                'word_count': word_count,
                'char_count': char_count,
                'sentence_count': c['sentence_count'],
                'estimated_token_count': int(word_count * 1.3),
                'language': self.cfg.language,
                'reading_time_seconds': int((word_count / 200) * 60)
            }
            
            enriched_chunks.append({
                'chunk_id': meta['chunk_id'],
                'text': text,
                'metadata': meta
            })
        return enriched_chunks

## 12. Dataset Validation
Automatically detects duplicate chunks, near duplicates (using Rapidfuzz), empty chunks, and invalid formatting.

In [12]:
class DatasetValidator:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.seen_hashes = set()
        self.seen_texts = []
        self.drop_stats = {'empty': 0, 'too_short': 0, 'too_long': 0, 'exact_dup': 0, 'near_dup': 0, 'invalid_unicode': 0}
        
    def validate(self, chunks: List[Dict]) -> List[Dict]:
        valid = []
        for c in chunks:
            text = c['text']
            wc = c['metadata']['word_count']
            
            if not text.strip():
                self.drop_stats['empty'] += 1; continue
            if wc < self.cfg.min_words_per_chunk:
                self.drop_stats['too_short'] += 1; continue
            if wc > self.cfg.max_words_per_chunk:
                self.drop_stats['too_long'] += 1; continue
            if '\ufffd' in text:
                self.drop_stats['invalid_unicode'] += 1; continue
                
            chash = c['metadata']['chunk_hash']
            if chash in self.seen_hashes:
                self.drop_stats['exact_dup'] += 1; continue
                
            # Near duplicate detection (expensive but requested)
            is_near_dup = False
            # Optimization: only compare with last 50 seen texts to save extreme O(N^2) time
            for seen_t in self.seen_texts[-50:]:
                if fuzz.ratio(text, seen_t) > self.cfg.duplicate_threshold:
                    is_near_dup = True
                    break
            if is_near_dup:
                self.drop_stats['near_dup'] += 1; continue
                
            self.seen_hashes.add(chash)
            self.seen_texts.append(text)
            valid.append(c)
        return valid
        
    def report(self):
        console.print('[bold cyan]Validation Drops[/bold cyan]')
        for k, v in self.drop_stats.items():
            if v > 0: console.print(f"  - {k}: {v}")

## 13. Dataset Creation (Multi-Schema & Synthetic)
Generates multiple dataset formats (Instruction, Conversation, Raw Text). Optional synthetic QA pairs using local CPU Hugging Face model.

In [13]:
class DatasetCreator:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.generator = None
        try:
            from transformers import pipeline
            self.generator = pipeline('text2text-generation', model='google/flan-t5-small', device=-1)
            console.print('[dim]Local CPU model loaded for synthetic QA.[/dim]')
        except Exception:
            console.print('[dim]Synthetic generation skipped (model unavailable).[/dim]')
            
    def create(self, valid_chunks: List[Dict]) -> List[Dict]:
        dataset = []
        for c in tqdm(valid_chunks, desc='Building Schemas'):
            text = c['text']
            meta = c['metadata']
            
            instruction = f"Summarize the key points from the section '{meta['hierarchy_path']}' in '{meta['book']}'."
            
            if self.generator and meta['word_count'] > 50:
                try:
                    prompt = f"Generate a question based on this text: {text}\n\nQuestion:"
                    res = self.generator(prompt, max_new_tokens=40, do_sample=False)
                    instruction = res[0]['generated_text']
                except:
                    pass
                    
            dataset.append({
                'id': c['chunk_id'],
                'metadata': meta,
                'schemas': {
                    'raw_text': {'text': text},
                    'alpaca': {'instruction': instruction, 'input': text, 'output': text},
                    'chatml': {'messages': [
                        {'role': 'system', 'content': 'You are a knowledge assistant.'},
                        {'role': 'user', 'content': f"Context: {text}\n\nTask: {instruction}"},
                        {'role': 'assistant', 'content': text}
                    ]}
                }
            })
        return dataset

## 14. Dataset Statistics
Generates token estimates, language distribution, size charts.

In [14]:
class StatisticsGenerator:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        
    def generate(self, dataset: List[Dict]) -> Dict:
        if not dataset: return {}
        df = pd.DataFrame([d['metadata'] for d in dataset])
        
        stats = {
            'total_chunks': len(df),
            'total_words': int(df['word_count'].sum()),
            'total_sentences': int(df['sentence_count'].sum()),
            'total_tokens': int(df['estimated_token_count'].sum()),
            'avg_chunk_size_words': float(df['word_count'].mean()),
            'largest_chunk_words': int(df['word_count'].max()),
            'smallest_chunk_words': int(df['word_count'].min()),
            'books_processed': int(df['book'].nunique())
        }
        
        with open(self.cfg.stats_dir / 'dataset_stats.json', 'wb') as f:
            f.write(orjson.dumps(stats, option=orjson.OPT_INDENT_2))
            
        plt.figure(figsize=(10, 5))
        plt.hist(df['word_count'], bins=30, color='#3b82f6')
        plt.title('Chunk Size Distribution (Words)')
        plt.savefig(self.cfg.stats_dir / 'word_count_dist.png')
        plt.close()
        
        return stats

## 15. Export & Summary Report
Writes all data to disk in JSONL, CSV, Parquet, and Arrow formats. Saves raw texts, chunks, and an HTML report.

In [15]:
class Exporter:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        
    def export(self, dataset: List[Dict], stats: Dict):
        if not dataset: return
        
        # Save raw chunks
        with open(self.cfg.chunks_dir / 'raw_chunks.jsonl', 'wb') as f:
            for item in dataset:
                f.write(orjson.dumps({'id': item['id'], 'text': item['schemas']['raw_text']['text']}) + b'\n')
                
        # Export Alpaca JSONL
        alpaca_path = self.cfg.exports_dir / 'alpaca_dataset.jsonl'
        with open(alpaca_path, 'wb') as f:
            for item in dataset:
                f.write(orjson.dumps(item['schemas']['alpaca']) + b'\n')
                
        # Export ChatML JSONL
        chatml_path = self.cfg.exports_dir / 'chatml_dataset.jsonl'
        with open(chatml_path, 'wb') as f:
            for item in dataset:
                f.write(orjson.dumps(item['schemas']['chatml']) + b'\n')
                
        # Flatten for CSV & Parquet
        flat_data = []
        for d in dataset:
            flat_data.append({
                'id': d['id'],
                'book': d['metadata']['book'],
                'instruction': d['schemas']['alpaca']['instruction'],
                'text': d['schemas']['alpaca']['input'],
                'word_count': d['metadata']['word_count']
            })
            
        df = pd.DataFrame(flat_data)
        df.to_csv(self.cfg.exports_dir / 'dataset.csv', index=False)
        hf_ds = Dataset.from_pandas(df)
        hf_ds.to_parquet(self.cfg.exports_dir / 'dataset.parquet')
        
        self._build_html_report(stats)
        console.print(f'[bold green]✓ Exported to {self.cfg.exports_dir}[/bold green]')
        
    def _build_html_report(self, stats: Dict):
        html = f"""
        <html><body>
        <h1>Pipeline Summary</h1>
        <ul>
            <li>Books Processed: {stats.get('books_processed',0)}</li>
            <li>Total Valid Chunks: {stats.get('total_chunks',0)}</li>
            <li>Total Words: {stats.get('total_words',0)}</li>
            <li>Total Tokens: {stats.get('total_tokens',0)}</li>
        </ul>
        <img src="../statistics/word_count_dist.png"/>
        </body></html>
        """
        with open(self.cfg.reports_dir / 'summary.html', 'w') as f:
            f.write(html)

## 16. Pipeline Execution
Orchestrates the pipeline using `concurrent.futures.ProcessPoolExecutor` to process multiple PDFs concurrently on CPU.

In [16]:
def process_single_pdf(pdf_info: Dict, cfg: PipelineConfig) -> List[Dict]:
    meta_ext = MetadataExtractor()
    struct_ext = StructureExtractor(cfg)
    cleaner = TextCleaner()
    segmenter = DocumentSegmenter()
    chunker = SemanticChunker(cfg)
    meta_gen = MetadataGenerator(cfg)
    
    doc_meta = meta_ext.extract(pdf_info)
    blocks = struct_ext.extract(pdf_info['path'])
    segmented = segmenter.segment(blocks, cleaner, doc_meta['book_title'])
    raw_chunks = chunker.chunk(segmented)
    return meta_gen.generate(raw_chunks, doc_meta)

def run_pipeline():
    console.print('[bold magenta]🚀 Starting PDF-to-Instruction Pipeline (CPU Local)[/bold magenta]\n')
    start_time = time.time()
    
    setup_environment(config)
    scanner = PDFScanner(config)
    valid_pdfs = scanner.scan()
    
    if not valid_pdfs:
        console.print('[bold red]Pipeline aborted. No valid PDFs found.[/bold red]')
        return
        
    all_chunks = []
    # Multithreading (Jupyter-safe alternative to multiprocessing)
    with concurrent.futures.ThreadPoolExecutor(max_workers=config.max_workers) as executor:
        futures = {executor.submit(process_single_pdf, pdf, config): pdf for pdf in valid_pdfs}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(valid_pdfs), desc="Processing PDFs"):
            try:
                chunks = future.result()
                all_chunks.extend(chunks)
            except Exception as e:
                logger.error(f"Error processing PDF: {e}")
                
    validator = DatasetValidator(config)
    valid_chunks = validator.validate(all_chunks)
    validator.report()
    
    creator = DatasetCreator(config)
    dataset = creator.create(valid_chunks)
    
    stats_gen = StatisticsGenerator(config)
    stats = stats_gen.generate(dataset)
    
    exporter = Exporter(config)
    exporter.export(dataset, stats)
    
    duration = time.time() - start_time
    console.print(f"\n[bold bright_green]✅ Pipeline Completed in {duration:.2f} seconds![/bold bright_green]")

if __name__ == '__main__':
    run_pipeline()

🚀 Starting PDF-to-Instruction Pipeline (CPU Local)

✓ Pipeline directories established.

Scanning PDFs:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Discovery: Found 2 valid unique PDFs out of 3 total.

Processing PDFs:   0%|          | 0/2 [00:00<?, ?it/s]

...

Building Schemas:   0%|          | 0/620 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (518 > 512). Running this sequence through the model will result in indexing errors


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✓ Exported to pipeline_workspace/exports

✅ Pipeline Completed in 361.72 seconds!